In [ ]:
!pip -q install -U transformers datasets accelerate evaluate

import json, torch
from datasets import Dataset
import numpy as np
import evaluate
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 52.1 MB/s eta 0:00:00


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
MODEL_NAME = "klue/roberta-base"  # 필요하면 바꾸기

label2id = {"red": 0, "blue": 1}
id2label = {0: "red", 1: "blue"}

def load_json_list(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    # label 문자열 -> id
    for x in data:
        x["labels"] = label2id[x["label"]]
    return Dataset.from_list(data)

train_ds = load_json_list("train.json")
valid_ds = load_json_list("valid.json")
test_ds  = load_json_list("test.json")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

In [ ]:
def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,   # 필요시 256
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
valid_ds = valid_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

acc = evaluate.load("accuracy")
f1  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": acc.compute(predictions=preds, references=labels)["accuracy"],
        "macro_f1": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

args = TrainingArguments(
    output_dir="/content/out",
    eval_strategy="epoch",   # <- 여기
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    num_train_epochs=6,
    weight_decay=0.01,
    warmup_ratio=0.06,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    report_to="none",
)

Map:   0%|          | 0/17710 [00:00<?, ? examples/s]

Map:   0%|          | 0/2214 [00:00<?, ? examples/s]

Map:   0%|          | 0/2214 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
drop_cols = [c for c in ["label", "text"] if c in train_ds.column_names]
train_ds = train_ds.remove_columns(drop_cols)

drop_cols = [c for c in ["label", "text"] if c in valid_ds.column_names]
valid_ds = valid_ds.remove_columns(drop_cols)

drop_cols = [c for c in ["label", "text"] if c in test_ds.column_names]
test_ds = test_ds.remove_columns(drop_cols)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate(test_ds)

trainer.save_model("/content/political_redblue_cls")
tokenizer.save_pretrained("/content/political_redblue_cls")

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.682896,0.675469,0.575881,0.461892
2,0.613577,0.684056,0.594851,0.585124
3,0.503217,0.646019,0.660795,0.656863
4,0.369725,0.763296,0.661247,0.657868
5,0.257271,0.889532,0.671635,0.668739
6,0.183754,0.993652,0.677055,0.675295


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/political_redblue_cls/tokenizer_config.json',
 '/content/political_redblue_cls/tokenizer.json')

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = "/content/political_redblue_cls"

tok = AutoTokenizer.from_pretrained(MODEL_DIR)
mdl = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval().cuda()

@torch.inference_mode()
def predict_proba(texts):
    if isinstance(texts, str):
        texts = [texts]
    batch = tok(texts, return_tensors="pt", truncation=True, padding=True, max_length=256)
    batch = {k: v.cuda() for k, v in batch.items()}
    out = mdl(**batch)
    probs = torch.softmax(out.logits, dim=-1).detach().cpu().numpy()
    # probs[:,0]=red, probs[:,1]=blue
    return [{"red": float(p[0]), "blue": float(p[1])} for p in probs]

predict_proba("천안함 사건을 기억해야 합니다")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'red': 0.20719605684280396, 'blue': 0.792803943157196}]

In [ ]:
SAVE_DIR = "/content/redblue_cls"

trainer.save_model(SAVE_DIR)          # model + config 저장
tokenizer.save_pretrained(SAVE_DIR)   # tokenizer 저장

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/redblue_cls/tokenizer_config.json',
 '/content/redblue_cls/tokenizer.json')

In [ ]:
!ls -al /content/redblue_cls


total 432892
drwxr-xr-x 2 root root      4096 Jan 27 07:54 .
drwxr-xr-x 1 root root      4096 Jan 27 07:54 ..
-rw-r--r-- 1 root root       914 Jan 27 07:54 config.json
-rw-r--r-- 1 root root 442502720 Jan 27 07:54 model.safetensors
-rw-r--r-- 1 root root       427 Jan 27 07:54 tokenizer_config.json
-rw-r--r-- 1 root root    752097 Jan 27 07:54 tokenizer.json
-rw-r--r-- 1 root root      5201 Jan 27 07:54 training_args.bin


In [ ]:
!zip -r redblue_cls.zip /content/redblue_cls
from google.colab import files
files.download("redblue_cls.zip")

  adding: content/redblue_cls/ (stored 0%)
  adding: content/redblue_cls/tokenizer_config.json (deflated 49%)
  adding: content/redblue_cls/tokenizer.json (deflated 69%)
  adding: content/redblue_cls/model.safetensors (deflated 10%)
  adding: content/redblue_cls/training_args.bin (deflated 53%)
  adding: content/redblue_cls/config.json (deflated 52%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>